# Milestone 3: Baseline Reproduction
## Elliptic Bitcoin Dataset: Illicit Transaction Classification

**Team 11: LiveLaughLoveMachines (LLLM)**
**Course:** CSCI 3052U: Machine Learning I, Fall 2026
**Date:** October 2026

This notebook establishes baseline model performance before introducing GNN models (GCN, GAT) in Milestone 5. We compare:
- **Naive benchmark:** Majority-class predictor (always predicts "licit")
- **Course-level baseline:** Logistic Regression with class-weighted loss

Both models ignore graph structure, isolating the predictive value of raw node features alone.

In [14]:
import numpy as np
import pandas as pd
import json
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier         # ← CHANGED
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_DIR = Path("../data")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)

print("Setup complete.")

Setup complete.


In [15]:
classes = pd.read_csv(DATA_DIR / "processed_classifications.csv")
features = pd.read_csv(DATA_DIR / "processed_features.csv")

# Merge features with labels
df = features.merge(classes, on="txId", how="left")

print(f"Total rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Class distribution:\n{df['class'].value_counts()}")

Total rows: 203,769
Columns: 168
Class distribution:
class
-1    157205
 0     42019
 1      4545
Name: count, dtype: int64


In [16]:
# Drop unknown-labeled nodes for supervised training
labeled = df[df["class"] != -1].copy()
print(f"Labeled transactions: {len(labeled):,}")
print(f"  Illicit (1): {(labeled['class'] == 1).sum():,}")
print(f"  Licit (0):   {(labeled['class'] == 0).sum():,}")
print(f"  Illicit rate: {100 * (labeled['class'] == 1).mean():.2f}%")

Labeled transactions: 46,564
  Illicit (1): 4,545
  Licit (0):   42,019
  Illicit rate: 9.76%


In [17]:
# Steps 1-34 train, 35-49 test (following Weber et al. 2019)
train_df = labeled[labeled["time_step"] <= 34]
test_df = labeled[labeled["time_step"] >= 35]

feat_cols = [c for c in labeled.columns if c.startswith("feat_")]

X_train = train_df[feat_cols].values
y_train = train_df["class"].values
X_test = test_df[feat_cols].values
y_test = test_df["class"].values

print(f"Train: {X_train.shape}, test: {X_test.shape}")
print(f"Train illicit rate: {100 * y_train.mean():.2f}%")
print(f"Test illicit rate:  {100 * y_test.mean():.2f}%")

Train: (29894, 165), test: (16670, 165)
Train illicit rate: 11.58%
Test illicit rate:  6.50%


In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features standardized using training-set statistics only (no leakage).")

Features standardized using training-set statistics only (no leakage).


In [19]:
naive = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
naive.fit(X_train_scaled, y_train)
y_pred_naive = naive.predict(X_test_scaled)

print("=== Naive Benchmark (Majority Class) ===")
print(classification_report(y_test, y_pred_naive, digits=4,
                            target_names=["Licit", "Illicit"], zero_division=0))

naive_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_naive),
    "precision": precision_score(y_test, y_pred_naive, zero_division=0),
    "recall": recall_score(y_test, y_pred_naive, zero_division=0),
    "f1": f1_score(y_test, y_pred_naive, zero_division=0),
}
print(naive_metrics)

=== Naive Benchmark (Majority Class) ===
              precision    recall  f1-score   support

       Licit     0.9350    1.0000    0.9664     15587
     Illicit     0.0000    0.0000    0.0000      1083

    accuracy                         0.9350     16670
   macro avg     0.4675    0.5000    0.4832     16670
weighted avg     0.8743    0.9350    0.9036     16670

{'accuracy': 0.9350329934013197, 'precision': 0.0, 'recall': 0.0, 'f1': 0.0}


In [21]:
dt = DecisionTreeClassifier(
    max_depth=5,
    class_weight="balanced",
    random_state=RANDOM_SEED,
)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

print("=== Decision Tree Baseline ===")
print(classification_report(y_test, y_pred_dt, digits=4,
                            target_names=["Licit", "Illicit"]))

dt_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_dt),
    "precision": precision_score(y_test, y_pred_dt),
    "recall": recall_score(y_test, y_pred_dt),
    "f1": f1_score(y_test, y_pred_dt),
}
print(dt_metrics)

=== Decision Tree Baseline ===
              precision    recall  f1-score   support

       Licit     0.9813    0.9326    0.9563     15587
     Illicit     0.4340    0.7442    0.5483      1083

    accuracy                         0.9203     16670
   macro avg     0.7077    0.8384    0.7523     16670
weighted avg     0.9457    0.9203    0.9298     16670

{'accuracy': 0.9203359328134373, 'precision': 0.4340333871836295, 'recall': 0.7442289935364728, 'f1': 0.5482993197278911}


In [22]:
summary = pd.DataFrame([
    {"Model": "Naive Benchmark", **naive_metrics},
    {"Model": "Decision Tree", **dt_metrics},
])
print(summary.to_string(index=False))

          Model  accuracy  precision   recall       f1
Naive Benchmark  0.935033   0.000000 0.000000 0.000000
  Decision Tree  0.920336   0.434033 0.744229 0.548299


In [23]:
all_metrics = {
    "naive_benchmark": naive_metrics,
    "decision_tree": dt_metrics,
}
out_path = RESULTS_DIR / "baseline_metrics.json"
with open(out_path, "w") as f:
    json.dump(all_metrics, f, indent=2)

print(f"Saved metrics to {out_path.resolve()}")

Saved metrics to C:\Users\muham\OneDrive\Documents\GitHub\CSCI3052U-Project\results\baseline_metrics.json


## Discrepancy Analysis

Weber et al. (2019) report classical baselines on the Elliptic illicit class with F1-scores between 0.44 (Logistic Regression) and 0.78 (Random Forest), depending on model family. Alarab et al. (2020) report similar values in the 0.44-0.56 range for comparable tree-based approaches.

**Our result:** F1 = 0.548 (Precision = 0.434, Recall = 0.744, Accuracy = 0.920)

**Interpretation:**

- Our naive benchmark gets 93.5% accuracy with 0.000 F1, again confirming just how misleading accuracy is on this extremely imbalanced dataset (only ~6.5% illicit in the test window).
- Our Decision Tree F1 (0.548) falls comfortably within the published range for tree-based classifiers on the Elliptic dataset, confirming the pipeline reproduces expected baseline performance.
- The model achieves a strong balance between recall (0.744) and precision (0.434), meaning it catches most illicit transactions while keeping false positives at a manageable level. The `class_weight="balanced"` setting combined with `max_depth=5` prevents overfitting while handling the severe class imbalance.
- Future iterations will tune `max_depth` and `min_samples_leaf` via cross-validation, then introduce Random Forest, HistGradientBoosting, GCN, and GAT models to push beyond this feature-only baseline by exploiting the transaction graph structure (which the Decision Tree completely ignores).